# Getting Started with Claude Sonnet 5.5 on Amazon Bedrock

**Claude Sonnet 5.5 is a smarter, more efficient Sonnet. It is a clear leap forward from Sonnet 5 on coding and knowledge work, at a much lower cost per task.**

Sonnet 5.5 is the model you reach for when the approach is already clear and what's left is to execute it quickly, reliably, and at scale. It is Anthropic's strongest Sonnet model for coding, handles well-defined agent tasks on its own or as a subagent, and brings improvements across enterprise knowledge work such as analysis, documents, and slide decks, with clearer, more direct writing that needs less cleanup.

This notebook shows how to call Sonnet 5.5 on Amazon Bedrock. The primary path throughout is the **Anthropic Messages API** via the `anthropic` SDK's `AnthropicBedrock` client, which targets the standard `bedrock-runtime` endpoint. Each section also shows the equivalent call on the **Converse** and **InvokeModel** APIs so you can pick the one that fits your stack. Every cell runs live against the model.

---

## Why Claude Sonnet 5.5 on Amazon Bedrock

Amazon Bedrock gives you Sonnet 5.5 while keeping your data within AWS infrastructure with regional data residency. Prompts and outputs are not shared with model providers and are not used to train models. You work with the AWS controls your team already uses:

- **AWS IAM** for access control
- **AWS CloudTrail** for audit
- **Amazon CloudWatch** for monitoring
- **Amazon Bedrock Guardrails** for content safety

Usage appears on your AWS bill.

---

## What's new in Claude Sonnet 5.5

- **Lower cost per task**: A clear leap over Sonnet 5 on coding and knowledge work, at a much lower cost per task. The unit that matters is cost per task, not price per token.
- **Strongest Sonnet for coding**: Builds features, reviews code, fixes bugs, and explains its changes clearly. Built for everyday development at volume on defined coding tasks.
- **Agentic reliability**: Takes on well-defined agent tasks on its own or as a subagent (investigation, review, and drafting) at a cost that makes it practical to run many agents in parallel.
- **Adaptive thinking**: Extended reasoning is always on, with `high` as the default effort. Control depth with effort levels; it cannot be turned off.
- **Clearer enterprise output**: Improvements across analysis, documents, and slide decks, with writing that needs less cleanup.

---

## Key capabilities

| Capability | Details |
|------------|---------|
| **Context window** | 1M tokens input |
| **Max output** | 128K tokens |
| **Adaptive thinking** | Always on; `high` effort by default. Effort levels: `between_tools`, `low`, `medium`, `high`, `xhigh`, `max`. |
| **Vision** | High-resolution image input for screenshots, diagrams, charts, and design files |
| **Languages** | English, French, Modern Standard Arabic, Mandarin Chinese, Hindi, Spanish, Portuguese, Korean, Japanese, German, Russian, Polish, and others |

---

## Model ID and API options

Claude Sonnet 5.5 is available on `bedrock-runtime` through the **Global cross-Region inference profile**. Use the model ID `global.anthropic.claude-sonnet-5-5` on all three APIs.

| API | How you call it | When to use |
|-----|-----------------|-------------|
| **Messages API** (primary) | `AnthropicBedrock` client from the `anthropic` SDK | Recommended. Anthropic's native request/response shape with full feature support. Advanced fields pass through `extra_body`. |
| **InvokeModel** | boto3 `bedrock-runtime`, native Anthropic JSON body | When you want the raw request body or cannot add the `anthropic` SDK dependency. |
| **Converse** | boto3 `bedrock-runtime` `converse` | Bedrock's unified API for the simplest cross-model integration. |

See the [Amazon Bedrock documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/models-supported.html) for the full list of supported Regions.

---

## Breaking changes from Sonnet 5

Everything below either behaves differently or is not available on Bedrock, relative to Sonnet 5. Each item is demonstrated in the section noted.

| Change | Impact |
|--------|--------|
| **Extended thinking removed** | `thinking: {"type": "enabled", "budget_tokens": N}` returns a **400 error**. Use adaptive thinking with effort levels instead (Section 3). |
| **Thinking cannot be disabled** | `thinking: {"type": "disabled"}` returns a **400 error**. The lowest setting is `between_tools` (Section 3). |
| **Sampling parameters removed** | `temperature`, `top_p`, or `top_k` set to any value returns a **400 error**. Remove them from all calls. |
| **Native structured outputs not on Bedrock** | `response_format: {"type": "json_schema"}` returns a **400 error** (`Extra inputs are not permitted`). Request JSON via prompting instead (Section 7). |
| **No server-side model fallback on Bedrock** | Automatic fallback on a refusal is a Claude API feature and is not available on Bedrock. Handle `stop_reason: "refusal"` and retry on another model yourself (Section 11). |
| **Model ID** | Use `global.anthropic.claude-sonnet-5-5`. |

## 1. Setup and configuration

**Prerequisites:**
- An AWS account with Amazon Bedrock access to Claude Sonnet 5.5
- AWS credentials configured (for example via `aws configure` or SSO)
- Python 3.10+
- IAM permissions: `bedrock:InvokeModel`, `bedrock:InvokeModelWithResponseStream`
- Packages: `boto3` and `anthropic[bedrock]` (the Messages API client used throughout)

In [ ]:
# Install required packages (uncomment if needed)
#!pip install boto3 "anthropic[bedrock]" --upgrade 2>&1 | tail -3

In [ ]:
import json
import base64
import re
import boto3
from botocore.config import Config
from anthropic import AnthropicBedrock

# Region and model
REGION = "us-east-1"
MODEL_ID = "global.anthropic.claude-sonnet-5-5"
ANTHROPIC_VERSION = "bedrock-2023-05-31"

# Primary client: Anthropic Messages API over the standard bedrock-runtime endpoint.
client = AnthropicBedrock(aws_region=REGION)

# Secondary client: boto3 bedrock-runtime for the Converse and InvokeModel examples.
# A longer read timeout keeps high-effort/compaction responses from timing out;
# retries disabled so failures surface immediately. Both are supported botocore Config keys.
BEDROCK_CONFIG = Config(read_timeout=1200, retries={"max_attempts": 1})
bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION, config=BEDROCK_CONFIG)

print(f"Region:   {REGION}")
print(f"Model ID: {MODEL_ID}")
print(f"Messages API endpoint: {client.base_url}")
print("Setup complete.")

---

## 2. Basic usage

The Messages API is the primary way to call Sonnet 5.5 here. It uses Anthropic's native request/response format and is authenticated with AWS SigV4 by the `AnthropicBedrock` client. Sonnet 5.5 may return a `thinking` block before the `text` block, so select the text block rather than assuming index 0.

**Important:** Do NOT include `temperature`, `top_p`, or `top_k`. Sonnet 5.5 rejects non-default sampling parameters with a 400 error.

In [ ]:
question = (
    "What are three key considerations when designing "
    "a distributed system for high availability?"
)

# Messages API (primary)
print("Messages API:\n")
try:
    message = client.messages.create(
        model=MODEL_ID,
        max_tokens=2048,
        messages=[{"role": "user", "content": question}],
    )
    text = next(b.text for b in message.content if b.type == "text")
    print(text)
    print(f"\nUsage: Input={message.usage.input_tokens}, Output={message.usage.output_tokens}")
except Exception as e:
    print(f"Error: {e}")

The same request on **InvokeModel** and **Converse**:

In [ ]:
# InvokeModel (native Anthropic JSON body)
print("InvokeModel:\n")
try:
    body = {
        "anthropic_version": ANTHROPIC_VERSION,
        "max_tokens": 2048,
        "messages": [{"role": "user", "content": question}],
    }
    r = bedrock_runtime.invoke_model(modelId=MODEL_ID, contentType="application/json",
                                     accept="application/json", body=json.dumps(body))
    res = json.loads(r["body"].read())
    print(next(b["text"] for b in res["content"] if b["type"] == "text")[:300] + " ...")
except Exception as e:
    print(f"Error: {e}")

# Converse (unified API)
print("\n\nConverse:\n")
try:
    r = bedrock_runtime.converse(
        modelId=MODEL_ID,
        messages=[{"role": "user", "content": [{"text": question}]}],
        inferenceConfig={"maxTokens": 2048},
    )
    print(r["output"]["message"]["content"][0]["text"][:300] + " ...")
except Exception as e:
    print(f"Error: {e}")

---

## 3. Adaptive thinking

Adaptive thinking lets Claude reason deeply when a task needs it and respond quickly when it doesn't. **In Sonnet 5.5, thinking is always on with `high` effort by default**, so you don't need to configure anything for most workloads.

**How it works:**
- `thinking.type` is `"adaptive"` by default, so Claude decides how much to think based on task complexity.
- Control depth with `output_config.effort`. On the Messages API, `output_config` is passed through `extra_body`.
- Thinking **cannot be turned off**. The lowest setting is `between_tools`, which does no extended thinking and returns short progress updates between tool calls. `thinking: {"type": "disabled"}` returns a 400 error.

### Effort levels

| Effort | Behavior | Best for |
|--------|----------|----------|
| **max** | Exhaustive analysis, cost secondary | Hardest problems |
| **xhigh** | Maximum reasoning depth | Most complex analytical work |
| **high** | Deep reasoning (default) | Complex tasks |
| **medium** | Balanced | Mixed workloads |
| **low** | Minimal thinking, prioritizes speed | Simple, cost-sensitive queries |
| **between_tools** | No extended thinking; brief progress updates between tool calls | The floor for agentic loops |

**Recommendation:** The default is `high`. Anthropic recommends starting at `medium` for most workloads and moving to `high` for agentic or memory-heavy work.

In [ ]:
# Messages API: output_config goes in extra_body
try:
    message = client.messages.create(
        model=MODEL_ID,
        max_tokens=16000,
        thinking={"type": "adaptive"},
        extra_body={"output_config": {"effort": "high"}},
        messages=[{"role": "user", "content": "What is the sum of the first 30 prime numbers? Show your work."}],
    )
    for block in message.content:
        if block.type == "thinking":
            print("[Thinking block returned]\n")
        elif block.type == "text":
            print(block.text)
    tt = getattr(message.usage, "output_tokens_details", None)
    thinking_tokens = getattr(tt, "thinking_tokens", 0) if tt else 0
    print(f"\nUsage: Input={message.usage.input_tokens}, "
          f"Output={message.usage.output_tokens}, Thinking={thinking_tokens}")
except Exception as e:
    print(f"Error: {e}")

On **InvokeModel**, the same fields (`thinking`, `output_config`) go directly in the request body:

In [ ]:
try:
    body = {
        "anthropic_version": ANTHROPIC_VERSION,
        "max_tokens": 16000,
        "thinking": {"type": "adaptive"},
        "output_config": {"effort": "high"},
        "messages": [{"role": "user", "content": "What is the sum of the first 30 prime numbers? Show your work."}],
    }
    r = bedrock_runtime.invoke_model(modelId=MODEL_ID, body=json.dumps(body))
    res = json.loads(r["body"].read())
    print(next(b["text"] for b in res["content"] if b["type"] == "text")[:400] + " ...")
except Exception as e:
    print(f"Error: {e}")

### Comparing effort levels

Different effort levels produce different reasoning depths and costs. Set effort explicitly only when you want something other than the `high` default.

In [ ]:
def test_effort_level(effort_level, prompt):
    try:
        message = client.messages.create(
            model=MODEL_ID,
            max_tokens=12000,
            thinking={"type": "adaptive"},
            extra_body={"output_config": {"effort": effort_level}},
            messages=[{"role": "user", "content": prompt}],
        )
        tt = getattr(message.usage, "output_tokens_details", None)
        thinking = getattr(tt, "thinking_tokens", 0) if tt else 0
        chars = sum(len(b.text) for b in message.content if b.type == "text")
        return {"effort": effort_level, "input": message.usage.input_tokens,
                "thinking": thinking, "output": message.usage.output_tokens, "chars": chars}
    except Exception as e:
        return {"effort": effort_level, "error": str(e)[:60]}

prompt = (
    "Prove that there are infinitely many prime numbers. Then explain why "
    "this proof technique cannot be used to prove the twin prime conjecture."
)

print(f"{'effort':<14}{'input':>7}{'thinking':>10}{'output':>8}{'chars':>8}")
print("-" * 47)
for effort in ["low", "medium", "high", "xhigh", "max"]:
    r = test_effort_level(effort, prompt)
    if "error" in r:
        print(f"{r['effort']:<14} error: {r['error']}")
    else:
        print(f"{r['effort']:<14}{r['input']:>7}{r['thinking']:>10}{r['output']:>8}{r['chars']:>8}")

print("\nHigher effort means deeper reasoning and more tokens. Match the budget to task difficulty:")
print("turn it up for genuinely hard problems, down on easy ones to save latency and cost.")

---

## 4. Streaming

Streaming delivers tokens as they are generated, reducing perceived latency for long responses.

In [ ]:
# Messages API streaming
print("Messages API streaming:\n")
try:
    with client.messages.stream(
        model=MODEL_ID,
        max_tokens=512,
        messages=[{"role": "user", "content": "List five best practices for API security."}],
    ) as stream:
        for text in stream.text_stream:
            print(text, end="", flush=True)
    print()
except Exception as e:
    print(f"Error: {e}")

The same on **InvokeModel** and **Converse** (note Converse `content` must be a list of blocks, not a bare string):

In [ ]:
# InvokeModel streaming
print("InvokeModel streaming:\n")
try:
    body = {
        "anthropic_version": ANTHROPIC_VERSION,
        "max_tokens": 512,
        "messages": [{"role": "user", "content": "List five best practices for container security."}],
    }
    response = bedrock_runtime.invoke_model_with_response_stream(modelId=MODEL_ID, body=json.dumps(body))
    for event in response["body"]:
        chunk = json.loads(event["chunk"]["bytes"])
        if chunk["type"] == "content_block_delta" and chunk["delta"]["type"] == "text_delta":
            print(chunk["delta"]["text"], end="", flush=True)
    print()
except Exception as e:
    print(f"Error: {e}")

# Converse streaming
print("\n\nConverse streaming:\n")
try:
    response = bedrock_runtime.converse_stream(
        modelId=MODEL_ID,
        messages=[{"role": "user", "content": [{"text": "List five best practices for IAM security."}]}],
        inferenceConfig={"maxTokens": 512},
    )
    for event in response["stream"]:
        if "contentBlockDelta" in event:
            delta = event["contentBlockDelta"].get("delta", {})
            if "text" in delta:
                print(delta["text"], end="", flush=True)
    print()
except Exception as e:
    print(f"Error: {e}")

---

## 5. Token efficiency and cost per task

Sonnet 5.5 completes comparable work with fewer tokens than Sonnet 5, and lower per-token prices stack on top of that. The unit that matters is **cost per task**, not price per token.

The cell below runs the same task on Sonnet 5 and Sonnet 5.5 at the same effort and reports input, thinking, and output tokens side by side, so you can measure the token delta on *your* workload. Run it on something representative of your production traffic; a single toy prompt is not a benchmark.

In [ ]:
BASELINE_MODEL_ID = "global.anthropic.claude-sonnet-5"     # prior generation
CURRENT_MODEL_ID = MODEL_ID                                # global.anthropic.claude-sonnet-5-5

def usage_for(model_id, prompt, effort="high", max_tokens=8000):
    message = client.messages.create(
        model=model_id,
        max_tokens=max_tokens,
        thinking={"type": "adaptive"},
        extra_body={"output_config": {"effort": effort}},
        messages=[{"role": "user", "content": prompt}],
    )
    tt = getattr(message.usage, "output_tokens_details", None)
    text = next((b.text for b in message.content if b.type == "text"), "")
    return {"input": message.usage.input_tokens, "output": message.usage.output_tokens,
            "thinking": getattr(tt, "thinking_tokens", None) if tt else None, "chars": len(text)}

task = ("A service moves 40 TB/month of S3 egress at $0.09/GB and 12 TB/month of cross-AZ transfer "
        "at $0.01/GB. Compute the monthly total, then propose the single change with the largest "
        "saving and quantify it. Show the arithmetic. Under 150 words.")

print(f"{'model':40}{'input':>7}{'thinking':>10}{'output':>8}{'chars':>7}")
for mid in [BASELINE_MODEL_ID, CURRENT_MODEL_ID]:
    try:
        u = usage_for(mid, task)
        print(f"{mid:40}{u['input']:>7}{str(u['thinking']):>10}{u['output']:>8}{u['chars']:>7}")
    except Exception as e:
        print(f"{mid:40} unavailable: {str(e)[:50]}")

print("\nFewer thinking/output tokens for an equally good answer is the efficiency gain;")
print("the price change is on top of it. Always compare on your own representative tasks.")

### 5.1 Task budgets (beta)

Task budgets set a token spending limit for an entire agentic loop, so autonomous runs stay cost-bounded. When the budget is approached, Claude wraps up gracefully rather than stopping mid-task.

- Set the limit via `output_config.task_budget`.
- **Minimum budget is 20,000 tokens**. A lower value returns a 400 error.
- **Requires beta header:** `anthropic_beta: ["task-budgets-2026-03-13"]`

In [ ]:
try:
    message = client.messages.create(
        model=MODEL_ID,
        max_tokens=4096,
        thinking={"type": "adaptive"},
        extra_body={
            "anthropic_beta": ["task-budgets-2026-03-13"],
            "output_config": {"effort": "high", "task_budget": {"type": "tokens", "total": 25000}},
        },
        messages=[{"role": "user", "content": (
            "Plan a market research project for a new fintech startup targeting small "
            "businesses. Include competitor analysis, market sizing, and go-to-market strategy."
        )}],
    )
    print(f"Stop reason: {message.stop_reason}")
    print(f"Output tokens used: {message.usage.output_tokens} (budget: 25,000)\n")
    for block in message.content:
        if block.type == "text":
            print(block.text[:800] + "..." if len(block.text) > 800 else block.text)
except Exception as e:
    print(f"Error: {e}")
    print("Note: Task budgets require the beta header and may not be available in all Regions.")

---

## 6. Prompt caching

Prompt caching reduces latency and cost when you repeatedly send the same large context (system prompts, documents, examples) alongside different user queries.

- Mark content cacheable with `"cache_control": {"type": "ephemeral"}`.
- The first request writes the cache; subsequent requests read from it.
- Cache reads are significantly cheaper than reprocessing the same tokens.

In [ ]:
large_system = "You are an expert software engineer.\n" + "\n".join(
    f"Rule {i}: Always write clean, well-documented, production-ready code. "
    f"Follow best practices for error handling, testing, and performance."
    for i in range(1, 400)
)
print(f"System prompt size: ~{len(large_system)} chars\n")

def cached_call(user_msg):
    message = client.messages.create(
        model=MODEL_ID,
        max_tokens=64,
        system=[{"type": "text", "text": large_system, "cache_control": {"type": "ephemeral"}}],
        messages=[{"role": "user", "content": user_msg}],
    )
    return message.usage

try:
    u1 = cached_call("What are Python anti-patterns to avoid?")
    write1 = getattr(u1, "cache_creation_input_tokens", 0) or 0
    print(f"Request 1: cache_write={write1}, cache_read={getattr(u1, 'cache_read_input_tokens', 0) or 0}")
    u2 = cached_call("How do I write effective unit tests?")
    read2 = getattr(u2, "cache_read_input_tokens", 0) or 0
    print(f"Request 2: cache_read={read2}")
    if read2 > 0:
        print(f"\nCache hit: {read2} tokens served from cache.")
except Exception as e:
    print(f"Error: {e}")

---

## 7. Structured outputs (JSON)

As noted in Breaking changes, native structured outputs (`response_format: json_schema`) are **not available on Amazon Bedrock**. A request that includes `response_format` returns a 400 error (`Extra inputs are not permitted`).

The reliable approach on Bedrock is to request JSON in the prompt. Sonnet 5.5 follows a concrete example shape reliably.

In [ ]:
prompt = (
    "List exactly 3 programming languages with their year of creation. "
    "Respond with ONLY a JSON object, no prose and no markdown code fences, "
    "matching this shape exactly:\n"
    '{"languages": [{"name": "Python", "year": 1991}]}'
)

def extract_json(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`")
        text = text[text.find("{"):text.rfind("}") + 1]
    return json.loads(text)

# Messages API
print("Messages API, JSON via prompting:\n")
try:
    message = client.messages.create(
        model=MODEL_ID, max_tokens=256,
        messages=[{"role": "user", "content": prompt}],
    )
    text = next(b.text for b in message.content if b.type == "text")
    print(json.dumps(extract_json(text), indent=2))
except Exception as e:
    print(f"Error: {e}")

# Converse
print("\n\nConverse, JSON via prompting:\n")
try:
    r = bedrock_runtime.converse(
        modelId=MODEL_ID,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 256},
    )
    print(json.dumps(extract_json(r["output"]["message"]["content"][0]["text"]), indent=2))
except Exception as e:
    print(f"Error: {e}")

---

## 8. Tool use

Tool use lets Claude call functions you define. The Messages API and InvokeModel use Anthropic's native tool format; Converse uses Bedrock's `toolConfig` format.

In [ ]:
# Messages API tool use
print("Messages API, tool use:\n")
try:
    tools = [{
        "name": "calculator",
        "description": "Performs basic arithmetic. Use for any math question.",
        "input_schema": {
            "type": "object",
            "properties": {"expression": {"type": "string", "description": "The math expression"}},
            "required": ["expression"],
        },
    }]
    message = client.messages.create(
        model=MODEL_ID, max_tokens=512, tools=tools,
        messages=[{"role": "user", "content": "What is 42 * 17? Use the calculator tool."}],
    )
    print(f"Stop reason: {message.stop_reason}")
    for block in message.content:
        if block.type == "tool_use":
            print(f"Tool: {block.name}, Input: {json.dumps(block.input)}")
except Exception as e:
    print(f"Error: {e}")

The same on **Converse**, using the Bedrock `toolConfig` format:

In [ ]:
print("Converse, tool use:\n")
try:
    tool_config = {"tools": [{"toolSpec": {
        "name": "calculator",
        "description": "Performs basic arithmetic.",
        "inputSchema": {"json": {
            "type": "object",
            "properties": {"expression": {"type": "string", "description": "Math expression"}},
            "required": ["expression"],
        }},
    }}]}
    r = bedrock_runtime.converse(
        modelId=MODEL_ID,
        messages=[{"role": "user", "content": [{"text": "What is 42 * 17? Use the calculator tool."}]}],
        inferenceConfig={"maxTokens": 512},
        toolConfig=tool_config,
    )
    print(f"Stop reason: {r.get('stopReason')}")
    for block in r["output"]["message"]["content"]:
        if block.get("toolUse"):
            tu = block["toolUse"]
            print(f"Tool: {tu['name']}, Input: {json.dumps(tu['input'])}")
except Exception as e:
    print(f"Error: {e}")

---

## 9. Context compaction (beta)

Context compaction enables "infinite context" for long-running conversations and agentic tasks by automatically summarizing older context as it approaches the context window limit.

- Enable it with a `compact_20260112` edit under `context_management.edits` (passed through `extra_body` on the Messages API).
- Set a `trigger` threshold (e.g. 100K input tokens); older context is summarized when input exceeds it.
- The response returns a `compaction` block. Pass it back in subsequent requests to maintain continuity.
- **Requires beta header:** `anthropic_beta: ["compact-2026-01-12"]`
- Supported on the Messages API and InvokeModel. Converse returns a validation error.

Compaction won't trigger on short conversations; it's designed for 100K+ token contexts.

In [ ]:
try:
    message = client.messages.create(
        model=MODEL_ID,
        max_tokens=1024,
        extra_body={
            "anthropic_beta": ["compact-2026-01-12"],
            "context_management": {"edits": [{
                "type": "compact_20260112",
                "trigger": {"type": "input_tokens", "value": 100000},
            }]},
        },
        messages=[{"role": "user", "content": (
            "Let's start a long-running conversation about building a complex "
            "e-commerce platform. What are the key architectural decisions to make first?"
        )}],
    )
    has_compaction = any(getattr(b, "type", None) == "compaction" for b in message.content)
    print(f"Compaction triggered: {has_compaction}")
    print("(Won't trigger on short conversations; designed for 100K+ token contexts)\n")
    for block in message.content:
        if block.type == "text":
            print(block.text[:800] + "..." if len(block.text) > 800 else block.text)
except Exception as e:
    print(f"Error: {e}")
    print("Note: Compaction requires the beta header.")

---

## 10. High-resolution vision

Claude Sonnet 5.5 supports high-resolution image input, with better accuracy on charts, dense documents, screenshots, and complex diagrams than Sonnet 5. This is particularly useful for financial analysis (dense filings and charts) and cybersecurity (screenshots and logs).

In [ ]:
import struct, zlib

def create_sample_chart_png():
    """Create a simple 200x100 PNG with colored bars (simulates a bar chart)."""
    w, h = 200, 100
    bar_colors = [(66, 133, 244), (234, 67, 53), (251, 188, 4), (52, 168, 83)]
    bar_heights = [80, 55, 70, 90]
    bar_width, gap, start_x = 30, 15, 20
    raw = b""
    for y in range(h):
        raw += b"\x00"
        for x in range(w):
            drawn = False
            for i, (bh, color) in enumerate(zip(bar_heights, bar_colors)):
                bx = start_x + i * (bar_width + gap)
                if bx <= x < bx + bar_width and y >= h - bh:
                    raw += bytes(color)
                    drawn = True
                    break
            if not drawn:
                raw += b"\xff\xff\xff"
    def chunk(ct, d):
        c = ct + d
        return struct.pack(">I", len(d)) + c + struct.pack(">I", zlib.crc32(c) & 0xFFFFFFFF)
    ihdr = struct.pack(">IIBBBBB", w, h, 8, 2, 0, 0, 0)
    return (b"\x89PNG\r\n\x1a\n" + chunk(b"IHDR", ihdr)
            + chunk(b"IDAT", zlib.compress(raw)) + chunk(b"IEND", b""))

sample_image = create_sample_chart_png()
image_b64 = base64.b64encode(sample_image).decode("utf-8")
print(f"Generated sample chart: {len(sample_image)} bytes")
print("(In production, use real high-res images to leverage Sonnet 5.5's vision)\n")

question = "Describe this chart. What type is it? What do the colors represent?"

# Messages API vision
print("Messages API, vision:\n")
try:
    message = client.messages.create(
        model=MODEL_ID, max_tokens=2048,
        messages=[{"role": "user", "content": [
            {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": image_b64}},
            {"type": "text", "text": question},
        ]}],
    )
    print(next(b.text for b in message.content if b.type == "text"))
except Exception as e:
    print(f"Error: {e}")

The same image on **Converse**, using the native bytes source:

In [ ]:
print("Converse, vision:\n")
try:
    r = bedrock_runtime.converse(
        modelId=MODEL_ID,
        messages=[{"role": "user", "content": [
            {"image": {"format": "png", "source": {"bytes": sample_image}}},
            {"text": question},
        ]}],
        inferenceConfig={"maxTokens": 2048},
    )
    print(r["output"]["message"]["content"][0]["text"])
except Exception as e:
    print(f"Error: {e}")

---

## 11. Handling refusals

Sonnet 5.5 applies safety classifiers in cybersecurity, AI R&D, and anti-distillation (biology safeguards are the same as Sonnet 5). When a request is declined, the API returns a normal response with `stop_reason: "refusal"`. It is not an error.

As noted in Breaking changes, server-side model fallback is a Claude API feature and is **not available on Amazon Bedrock**. To fall back on a refusal, retry the request on another Claude model yourself (the Anthropic SDKs provide refusal-fallback middleware for this pattern).

In [ ]:
def call_with_refusal_fallback(prompt, primary=MODEL_ID, fallback="global.anthropic.claude-sonnet-5"):
    """Retry on a fallback model when the primary returns stop_reason == 'refusal'."""
    def _call(model_id):
        return client.messages.create(
            model=model_id, max_tokens=512,
            messages=[{"role": "user", "content": prompt}],
        )

    message = _call(primary)
    if message.stop_reason == "refusal":
        print(f"Primary ({primary}) refused; retrying on fallback ({fallback})...")
        message = _call(fallback)
    return message

# A benign prompt returns end_turn; the fallback path only triggers on stop_reason == "refusal".
try:
    message = call_with_refusal_fallback("Explain the CAP theorem in two sentences.")
    print(f"Stop reason: {message.stop_reason}")
    print(next(b.text for b in message.content if b.type == "text"))
except Exception as e:
    print(f"Error: {e}")

---

## 12. Migrating from Sonnet 5

| Step | Action |
|------|--------|
| 1 | Update the model ID: `global.anthropic.claude-sonnet-5` to `global.anthropic.claude-sonnet-5-5` |
| 2 | Remove `temperature`, `top_p`, and `top_k` from all requests (they now return a 400 error) |
| 3 | Replace any `thinking: {"type": "enabled", ...}` with `thinking: {"type": "adaptive"}` plus `output_config: {"effort": "high"}` |
| 4 | Replace any `thinking: {"type": "disabled"}` with `thinking: {"type": "between_tools"}` (thinking cannot be turned off) |
| 5 | Handle `stop_reason: "refusal"` explicitly and retry on another model if needed (no server-side fallback on Bedrock) |
| 6 | Test with representative workloads; outputs may differ (improve) |

### Example: before and after

**Sonnet 5 request:**
```json
{
  "model": "global.anthropic.claude-sonnet-5",
  "thinking": {"type": "enabled", "budget_tokens": 10000},
  "temperature": 0.7,
  "messages": [...]
}
```

**Sonnet 5.5 request:**
```json
{
  "model": "global.anthropic.claude-sonnet-5-5",
  "thinking": {"type": "adaptive"},
  "output_config": {"effort": "high"},
  "messages": [...]
}
```

---

## 13. Next steps

- **Start with defaults**: adaptive thinking at `high` effort works for most workloads out of the box.
- **Tune effort levels**: drop to `medium`, `low`, or `between_tools` for simple/high-volume tasks; use `xhigh`/`max` for maximum depth.
- **Enable compaction** for long-running agents to avoid context window limits.
- **Set task budgets** for autonomous workflows to control cost.
- **Try high-res vision** with real screenshots, charts, and documents from your domain.

### Resources

- [Amazon Bedrock documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/)
- [Amazon Bedrock pricing](https://aws.amazon.com/bedrock/pricing/)
- [Claude documentation](https://platform.claude.com/docs)